### Configure Base Workflow Inputs

What this step does: configure base workflow inputs.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/enes/text-to-sign-production.git"
REPO_REF = "main"
BASE_RUN_NAME = "base_run"
RUN_QUALITATIVE_EXPORT = False
PANEL_SIZE = 8

WORKTREE_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")
print("Worktree root:", WORKTREE_ROOT)
print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Base run name:", BASE_RUN_NAME)


### Mount Google Drive

What this step does: mount google drive.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print("Drive mounted:", DRIVE_PROJECT_ROOT.parent)


### Acquire Repository

What this step does: acquire repository.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
%cd /content
_remove_output = !rm -rf "{WORKTREE_ROOT}"
_exit_code = getattr(_remove_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to remove existing worktree: {WORKTREE_ROOT}")
if WORKTREE_ROOT.exists():
    raise RuntimeError(f"Worktree still exists after cleanup: {WORKTREE_ROOT}")

_clone_output = !git clone "{REPO_URL}" "{WORKTREE_ROOT}"
_exit_code = getattr(_clone_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to clone repository from {REPO_URL} into {WORKTREE_ROOT}")
if not WORKTREE_ROOT.exists():
    raise RuntimeError(f"Repository clone did not create worktree: {WORKTREE_ROOT}")
if not (WORKTREE_ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"Cloned worktree is missing pyproject.toml: {WORKTREE_ROOT}")
if not (WORKTREE_ROOT / "src").is_dir():
    raise RuntimeError(f"Cloned worktree is missing src directory: {WORKTREE_ROOT}")

%cd {WORKTREE_ROOT}
_checkout_output = !git checkout "{REPO_REF}"
_exit_code = getattr(_checkout_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF} in {WORKTREE_ROOT}")

_revision_output = !git -C "{WORKTREE_ROOT}" rev-parse HEAD
_exit_code = getattr(_revision_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to verify checked-out revision in {WORKTREE_ROOT}")
if not (WORKTREE_ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"Checked-out worktree is missing pyproject.toml: {WORKTREE_ROOT}")
if not (WORKTREE_ROOT / "src").is_dir():
    raise RuntimeError(f"Checked-out worktree is missing src directory: {WORKTREE_ROOT}")
print("Repository ready:", WORKTREE_ROOT)


### Install And Import Package

What this step does: install and import package.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
%cd {WORKTREE_ROOT}
if not (WORKTREE_ROOT / "requirements-colab.txt").is_file():
    raise FileNotFoundError(f"Missing Colab requirements file: {WORKTREE_ROOT / 'requirements-colab.txt'}")
%pip install -r requirements-colab.txt

import sys
SRC_ROOT = WORKTREE_ROOT / "src"
if not SRC_ROOT.is_dir():
    raise RuntimeError(f"Repository post-check failed; missing src directory: {SRC_ROOT}")
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import text_to_sign_production as notebook_workflow
from text_to_sign_production.core.paths import ProjectLayout
from text_to_sign_production.core.files import (
    ensure_dir,
    require_dir,
    require_dir_contains,
    require_file,
    require_non_empty_file,
    verify_output_dir,
    verify_output_file,
)

runtime_layout = ProjectLayout(WORKTREE_ROOT)
drive_layout = ProjectLayout(DRIVE_PROJECT_ROOT)
if notebook_workflow.__name__ != "text_to_sign_production":
    raise RuntimeError("Imported unexpected package module")
if runtime_layout.data_root != WORKTREE_ROOT / "data":
    raise RuntimeError(f"Unexpected runtime data root: {runtime_layout.data_root}")
print("Package ready:", notebook_workflow.__name__)
print("Runtime data root:", runtime_layout.data_root)
print("Drive data root:", drive_layout.data_root)


### Restore Processed Training Inputs

What this step does: restore processed training inputs.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
MANIFEST_ARCHIVE = require_non_empty_file(
    drive_layout.processed_manifests_reports_archive(),
    label="Drive processed manifest/report archive",
)
TRAIN_SAMPLE_ARCHIVE = require_non_empty_file(
    drive_layout.processed_sample_archive("train"),
    label="Drive train processed sample archive",
)
VAL_SAMPLE_ARCHIVE = require_non_empty_file(
    drive_layout.processed_sample_archive("val"),
    label="Drive val processed sample archive",
)
for archive in (MANIFEST_ARCHIVE, TRAIN_SAMPLE_ARCHIVE, VAL_SAMPLE_ARCHIVE):
    %cd {WORKTREE_ROOT}
    _extract_output = !tar --zstd -xf "{archive}" -C "{WORKTREE_ROOT}"
    _exit_code = getattr(_extract_output, "exit_code", 0)
    if _exit_code != 0:
        raise RuntimeError(f"Failed to restore processed archive: {archive}")

TRAIN_MANIFEST_PATH = verify_output_file(runtime_layout.processed_manifest_path("train"), label="Train manifest")
VAL_MANIFEST_PATH = verify_output_file(runtime_layout.processed_manifest_path("val"), label="Validation manifest")
if TRAIN_MANIFEST_PATH.stat().st_size <= 0:
    raise RuntimeError(f"Train manifest is empty: {TRAIN_MANIFEST_PATH}")
if VAL_MANIFEST_PATH.stat().st_size <= 0:
    raise RuntimeError(f"Validation manifest is empty: {VAL_MANIFEST_PATH}")
TRAIN_SAMPLE_ROOT = require_dir_contains(runtime_layout.processed_samples_split_root("train"), "*.npz", label="Train samples")
VAL_SAMPLE_ROOT = require_dir_contains(runtime_layout.processed_samples_split_root("val"), "*.npz", label="Validation samples")
if not any(TRAIN_SAMPLE_ROOT.glob("*.npz")):
    raise FileNotFoundError(f"Train sample directory contains no .npz files: {TRAIN_SAMPLE_ROOT}")
if not any(VAL_SAMPLE_ROOT.glob("*.npz")):
    raise FileNotFoundError(f"Validation sample directory contains no .npz files: {VAL_SAMPLE_ROOT}")
print("Processed inputs restored:", TRAIN_MANIFEST_PATH, VAL_MANIFEST_PATH)


### Run Base Workflow

What this step does: run base workflow.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
from text_to_sign_production.workflows.base import (
    BaseWorkflowConfig,
    run_base_workflow,
    validate_base_inputs,
)
from text_to_sign_production.core.files import OutputExistsPolicy

base_config = BaseWorkflowConfig(
    project_root=WORKTREE_ROOT,
    run_name=BASE_RUN_NAME,
    run_qualitative_export=RUN_QUALITATIVE_EXPORT,
    panel_size=PANEL_SIZE,
    output_policy=OutputExistsPolicy.OVERWRITE,
)
validate_base_inputs(base_config)
base_result = run_base_workflow(base_config)
BASE_RUN_ROOT = verify_output_dir(runtime_layout.base_run_root(BASE_RUN_NAME), label="Base run root")
if base_result.run_root != BASE_RUN_ROOT:
    raise RuntimeError(f"Unexpected base run root: {base_result.run_root}")
print("Base workflow complete:", BASE_RUN_ROOT)


### Publish Base Run Archive

What this step does: publish base run archive.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
PUBLISHED_BASE_ARCHIVE = drive_layout.base_run_archive(BASE_RUN_NAME)
_mkdir_output = !mkdir -p "{PUBLISHED_BASE_ARCHIVE.parent}"
_exit_code = getattr(_mkdir_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to create Drive base archive directory: {PUBLISHED_BASE_ARCHIVE.parent}")
member = f"data/artifacts/base/{BASE_RUN_NAME}"
%cd {WORKTREE_ROOT}
_create_output = !tar --zstd -cf "{PUBLISHED_BASE_ARCHIVE}" -C "{WORKTREE_ROOT}" "{member}"
_exit_code = getattr(_create_output, "exit_code", 0)
if _exit_code != 0:
    raise RuntimeError(f"Failed to create base run archive: {PUBLISHED_BASE_ARCHIVE}")
verify_output_file(PUBLISHED_BASE_ARCHIVE, label="Published base run archive")
if PUBLISHED_BASE_ARCHIVE.stat().st_size <= 0:
    raise RuntimeError(f"Published base archive is empty: {PUBLISHED_BASE_ARCHIVE}")
print("Published base archive:", PUBLISHED_BASE_ARCHIVE)
